In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 512
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.3.18 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [ ]:
import os

for root, dirs, files in os.walk('/content'):
    for file in files:
        if file == "final_hateXplain.csv":
            print("Chemin complet à copier :", os.path.join(root, file))


Chemin complet à copier : /content/final_hateXplain.csv


In [ ]:
from datasets import load_dataset

# 1. Chargement du nouveau dataset
dataset = load_dataset("csv", data_files="final_hateXplain.csv", split="train")

# 2. Formatage simplifié (sans raisonnement)
def formatting_prompts_func(examples):
    # On ajuste l'instruction pour être directe
    instruction = "Determine if the following message contains hateful content. Answer only with 'True' or 'False'."
    texts = []

    for msg, lbl in zip(examples["comment"], examples["label"]):
        # Conversion du label HateXplain vers ton format
        clean_label = "True" if str(lbl).lower() == "hatespeech" else "False"

        # Structure ultra-simple pour Llama 3.2
        text = f"### Instruction:\n{instruction}\n\n### Input:\n{msg}\n\n### Response:\n{clean_label} <|end_of_text|>"
        texts.append(text)

    return { "text" : texts }

# On nettoie le dataset pour ne garder que la colonne "text"
dataset = dataset.map(formatting_prompts_func, batched = True, remove_columns=dataset.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/20109 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        warmup_steps = 100,

        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/20109 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,109 | Num Epochs = 3 | Total steps = 3,771
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 22,544,384 of 1,258,358,784 (1.79% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
20,3.702200
40,2.366300
60,2.132100
80,1.921500
100,1.977800
120,1.890000
140,1.862500
160,1.900200
180,1.861700
200,1.853900


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import pandas as pd
import torch
from tqdm import tqdm

# Avec 20 000 lignes, on passe à 200 exemples pour une meilleure fiabilité
eval_ds = dataset.shuffle(seed=42).select(range(200))

predictions = []
references = []

print("Analyse du modèle (Classification Directe True/False) en cours...")
FastLanguageModel.for_inference(model)

for entry in tqdm(eval_ds):
    # Extraction de la référence (ce qui est après Response:)
    # Le format est : "### Response:\nTrue <|end_of_text|>"
    true_label = entry['text'].split("### Response:\n")[-1].replace("<|end_of_text|>", "").strip().capitalize()

    # Préparation de l'entrée pour le modèle
    prompt_input = entry['text'].split("### Response:\n")[0] + "### Response:\n"
    inputs = tokenizer([prompt_input], return_tensors = "pt").to("cuda")

    # Génération courte (2 tokens suffisent pour True/False)
    outputs = model.generate(**inputs, max_new_tokens = 2, use_cache = True)

    # Extraction de la prédiction
    full_pred = tokenizer.batch_decode(outputs)[0].split("### Response:\n")[-1].strip().capitalize()

    # Nettoyage rapide pour ne garder que le premier mot au cas où
    pred_label = full_pred.split()[0].replace("<|end_of_text|>", "")

    predictions.append(pred_label)
    references.append(true_label)

print("\n" + "="*30)
print("RÉSULTATS DE VALIDATION (HateXplain Simple)")
print("="*30)

# On s'assure que les prédictions bizarres sont traitées comme 'False'
filtered_preds = [p if p in ["True", "False"] else "False" for p in predictions]

print(f"ACCURACY : {accuracy_score(references, filtered_preds):.2%}")
print(f"F1-SCORE : {f1_score(references, filtered_preds, pos_label='True'):.2%}")
print("\nRAPPORT DÉTAILLÉ :")
print(classification_report(references, filtered_preds))

print("\nMATRICE DE CONFUSION :")
conf_matrix = confusion_matrix(references, filtered_preds, labels=["False", "True"])
cm_df = pd.DataFrame(conf_matrix, index=['Attendu False', 'Attendu True'], columns=['Prédit False', 'Prédit True'])
print(cm_df)

Analyse du modèle (Classification Directe True/False) en cours...


100%|██████████| 200/200 [00:27<00:00,  7.19it/s]


RÉSULTATS DE VALIDATION (HateXplain Simple)
ACCURACY : 88.00%
F1-SCORE : 78.95%

RAPPORT DÉTAILLÉ :
              precision    recall  f1-score   support

       False       0.92      0.91      0.92       144
        True       0.78      0.80      0.79        56

    accuracy                           0.88       200
   macro avg       0.85      0.86      0.85       200
weighted avg       0.88      0.88      0.88       200


MATRICE DE CONFUSION :
               Prédit False  Prédit True
Attendu False           131           13
Attendu True             11           45


In [ ]:
test_cases = [
    {"text": "sick fuck number one i do not and never stick shit up my ass that all you freaks lmfao your funny gay nazis whitesupreamist", "expected": "False"},
    {"text": "We need to know about slavery.", "expected": "False"},
    {"text": "I love you.", "expected": "False"},
    {"text": "I hate rains.", "expected": "False"},
    {"text": "Kill all members of trans.", "expected": "True"}
]

FastLanguageModel.for_inference(model)

print("--- TEST DU MODÈLE (Classification Directe) ---")

# L'instruction DOIT être la même que celle de l'entraînement
instruction = "Determine if the following message contains hateful content. Answer only with 'True' or 'False'."

for case in test_cases:
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{case['text']}\n\n### Response:\n"

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # On réduit max_new_tokens car on ne veut qu'un mot
    outputs = model.generate(**inputs, max_new_tokens = 2, use_cache = True)

    # On récupère juste la réponse après le prompt
    prediction = tokenizer.batch_decode(outputs)[0].split("### Response:\n")[-1].strip().capitalize()

    # Nettoyage au cas où il y aurait des tokens résiduels
    prediction = prediction.split()[0].replace("<|end_of_text|>", "")

    status = "✅" if prediction == case['expected'] else "❌"

    print(f"{status} Input: {case['text']}")
    print(f"    Ref: {case['expected']} | Pred: {prediction}")
    print("-" * 50)


--- TEST DU MODÈLE (Classification Directe) ---
✅ Input: sick fuck number one i do not and never stick shit up my ass that all you freaks lmfao your funny gay nazis whitesupreamist
    Ref: False | Pred: False
--------------------------------------------------
✅ Input: We need to know about slavery.
    Ref: False | Pred: False
--------------------------------------------------
✅ Input: I love you.
    Ref: False | Pred: False
--------------------------------------------------
✅ Input: I hate rains.
    Ref: False | Pred: False
--------------------------------------------------
✅ Input: Kill all members of trans.
    Ref: True | Pred: True
--------------------------------------------------


In [ ]:
model.save_pretrained_gguf(
    "/content/drive/MyDrive/entrainement_final",
    tokenizer,
    quantization_method = "q4_k_m", # C'est le meilleur ratio poids/intelligence
)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:39<00:00, 39.17s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/entrainement_final`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/entrainement_final_gguf/Llama-3.2-1B.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/drive/MyDrive/entrainement_final_gguf/Llama-3.2-1B.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/Llama-3.2-1B'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/entrainement_final_gguf/Llama-3.2-1B.Q4_K_M.gguf -p "why is the sky blue?"


{'save_directory': '/content/drive/MyDrive/entrainement_final',
 'gguf_directory': '/content/drive/MyDrive/entrainement_final_gguf',
 'gguf_files': ['/content/drive/MyDrive/entrainement_final_gguf/Llama-3.2-1B.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
# Installation de llama-cpp-python avec support GPU (CUDA)
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.0/59.0 MB 14.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.8 MB/s eta 0:00:00


In [ ]:
from llama_cpp import Llama
import os

model_path = "/content/drive/MyDrive/entrainement_final_gguf/Llama-3.2-1B.Q4_K_M.gguf"

if not os.path.exists(model_path):
    print(f"Erreur : Le fichier n'a pas été trouvé à l'emplacement : {model_path}")
else:
    # On charge le modèle en utilisant le GPU s'il est disponible (n_gpu_layers=-1)
    llm = Llama(model_path=model_path, n_ctx=512, n_gpu_layers=-1)

    test_samples = [
        {"text": "I think everyone deserves respect regardless of their origin.", "expected": "False"},
        {"text": "They are taking our jobs and should be expelled!", "expected": "True"},
        {"text": "The weather is quite nice today for a walk.", "expected": "False"},
        {"text": "History shows that migration is a natural human process.", "expected": "False"},
        {"text": "I hate cilantro, it tastes like soap.", "expected": "False"},
        {"text": "We must teach children about the horrors of slavery.", "expected": "False"},
        {"text": "Trans people are like a virus.", "expected": "True"}
    ]

    # L'instruction simplifiée identique à l'entraînement
    instruction = "Determine if the following message contains hateful content. Answer only with 'True' or 'False'."

    print(f"\n--- Test de performance GGUF (Classification Directe) ---\n")

    for sample in test_samples:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{sample['text']}\n\n### Response:\n"

        output = llm(
            prompt,
            max_tokens=4, # Très court puisque c'est juste True ou False
            stop=["<|end_of_text|>", "###"],
            temperature=0.1 # On reste très "froid" pour plus de précision
        )

        # On récupère la réponse et on ne garde que le premier mot au cas où
        full_response = output['choices'][0]['text'].strip().capitalize()
        prediction = full_response.split()[0]

        status = "✅" if prediction == sample['expected'] else "❌"

        print(f"{status} Input: {sample['text']}")
        print(f"   Attendu: {sample['expected']} | Prédiction: {prediction}")
        print("-" * 50)

ModuleNotFoundError: No module named 'llama_cpp'